# residual-skip-add composite — cx27: stride-2 conv with a matching 1x1 skip downsample

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `conv-stride-downsample`, `residual-skip-add`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import torch.nn.functional as F

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "residual-skip-add"
DD_ATOM_IDS = ["conv-stride-downsample", "residual-skip-add"]
DD_SUBTOPICS = ["CNN: Stride downsample arithmetic", "CNN: Residual skip-connection add"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

When a ResNet block **downsamples** (output spatial size = input // 2) the main path uses a stride-2 conv. But then the **skip can't be a plain identity any more** — the input is `(N, C_in, H, W)` and the main-path output is `(N, C_out, H/2, W/2)`. They don't add.

The fix: project the skip through a **1x1 conv with stride 2** that matches both the channel change AND the spatial change. The shape arithmetic for stride-2:
- `H_out = floor((H + 2*pad - kernel) / stride) + 1`
- For `k=3, pad=1, stride=2`: `H_out = floor((H + 2 - 3) / 2) + 1 = floor((H - 1) / 2) + 1 = ceil(H/2)`.
- For `k=1, pad=0, stride=2`: `H_out = floor((H - 1) / 2) + 1 = ceil(H/2)`. SAME formula.

Both convs land on the same `(H_out, W_out)`, so the add type-checks. Channels must also match: skip projection is `Conv2d(C_in, C_out, kernel_size=1, stride=2, bias=False)`.

**Anatomy.**
```python
class DownBlock(nn.Module):
    def __init__(self, c_in, c_out):
        super().__init__()
        self.main = nn.Conv2d(c_in, c_out, 3, stride=2, padding=1, bias=False)
        self.skip = nn.Conv2d(c_in, c_out, 1, stride=2, bias=False)
    def forward(self, x):
        return F.relu(self.main(x) + self.skip(x))
```

**Why care.** ARENA's `BlockGroup` does exactly this at the start of each stage. Getting the skip downsample wrong is the most common ResNet shape mismatch.

### Composite Exercise — stride-2 conv with a matching 1x1 skip downsample

**Atoms exercised together**: `conv-stride-downsample`, `residual-skip-add`

Implement `cx27_make_downsample_block()` — return the class `DownBlock(nn.Module)`.

Required structure:
- `__init__(self, c_in, c_out)`:
  - `super().__init__()`
  - `self.main = nn.Conv2d(c_in, c_out, kernel_size=3, stride=2, padding=1, bias=False)`
  - `self.skip = nn.Conv2d(c_in, c_out, kernel_size=1, stride=2, padding=0, bias=False)`
- `forward(self, x)` returns `F.relu(self.main(x) + self.skip(x))`.

Spatial arithmetic: for input `(N, c_in, H, W)`, both `self.main(x)` and `self.skip(x)` produce `(N, c_out, ceil(H/2), ceil(W/2))`. They MUST line up — that's the whole point.

The test fuzzes several `(c_in, c_out, H, W)` combos and asserts: (1) output shape matches the stride-2 formula, (2) the skip is actually being added (zeroing the main's weight leaves the skip projection's output, NOT zero).

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

def cx27_make_downsample_block():
    """Return the DownBlock class."""
    raise NotImplementedError

def _test_cx27():
    DownBlock = cx27_make_downsample_block()
    assert issubclass(DownBlock, nn.Module)

    # Case A: instantiate and inspect the two child convs.
    t.manual_seed(0)
    block = DownBlock(c_in=4, c_out=8)
    assert isinstance(block.main, nn.Conv2d) and isinstance(block.skip, nn.Conv2d)
    assert block.main.stride == (2, 2), f'main must have stride=2, got {block.main.stride}'
    assert block.skip.stride == (2, 2), f'skip must have stride=2 to match, got {block.skip.stride}'
    assert block.main.kernel_size == (3, 3)
    assert block.skip.kernel_size == (1, 1), f'skip must be 1x1, got {block.skip.kernel_size}'
    assert block.main.in_channels == 4 and block.main.out_channels == 8
    assert block.skip.in_channels == 4 and block.skip.out_channels == 8

    # Case B: shape contract — stride-2 formula must match for several spatial sizes.
    import math
    for c_in, c_out, H, W in [(4, 8, 8, 8), (3, 6, 5, 7), (4, 8, 13, 11), (2, 4, 16, 16)]:
        blk = DownBlock(c_in, c_out)
        x = t.randn(2, c_in, H, W)
        out = blk(x)
        # k=3, pad=1, stride=2 and k=1, pad=0, stride=2 BOTH give ceil(H/2), ceil(W/2).
        expected_H = math.ceil(H / 2)
        expected_W = math.ceil(W / 2)
        assert tuple(out.shape) == (2, c_out, expected_H, expected_W), (
            f'cin={c_in} cout={c_out} H={H} W={W}: expected '
            f'(2,{c_out},{expected_H},{expected_W}), got {tuple(out.shape)}'
        )

    # Case C: residual-add is load-bearing — zero main, output must equal relu(skip(x)).
    block = DownBlock(c_in=3, c_out=5)
    x = t.randn(2, 3, 6, 6)
    with t.no_grad():
        block.main.weight.zero_()
    out = block(x)
    expected = F.relu(block.skip(x))
    assert t.allclose(out, expected, atol=1e-6), 'zeroed main: output should equal relu(skip(x))'
    # Also: out should NOT be all zero (skip is nonzero) — proves skip was added, not dropped.
    assert out.abs().sum().item() > 0, 'out all-zero — skip projection missing'

    # Case D: zero the SKIP — output should equal relu(main(x)).
    block = DownBlock(c_in=3, c_out=5)
    x = t.randn(2, 3, 6, 6)
    with t.no_grad():
        block.skip.weight.zero_()
    out = block(x)
    expected = F.relu(block.main(x))
    assert t.allclose(out, expected, atol=1e-6), 'zeroed skip: output should equal relu(main(x))'
    _dd_passed.add('cx27')

_test_cx27()

<details><summary>Show solution — cx27</summary>

```python
def cx27_make_downsample_block():
    class DownBlock(nn.Module):
        def __init__(self, c_in, c_out):
            super().__init__()
            # Atom A (conv-stride-downsample): main 3x3 stride-2 halves H,W (ceil rule).
            self.main = nn.Conv2d(c_in, c_out, kernel_size=3, stride=2, padding=1, bias=False)
            # Skip MUST land on the same (H/2, W/2). 1x1 stride-2 gives ceil(H/2), ceil(W/2).
            self.skip = nn.Conv2d(c_in, c_out, kernel_size=1, stride=2, padding=0, bias=False)

        def forward(self, x):
            # Atom B (residual-skip-add): add the (matched-shape) projections.
            return F.relu(self.main(x) + self.skip(x))

    return DownBlock
```

The trick is that two SEEMINGLY DIFFERENT conv configs (`k=3 p=1 s=2` and `k=1 p=0 s=2`) produce the same `ceil(H/2)` output spatial size. That's not a coincidence — it's the PyTorch stride formula `floor((H + 2p - k)/s) + 1` evaluating to the same expression for both. Drop the `padding=1` on main or change the skip kernel and the shapes diverge.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx27'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx27',
        'subtopics': ["CNN: Stride downsample arithmetic", "CNN: Residual skip-connection add"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()